In [23]:
from google import genai
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()

import vertexai 
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Tool,
    HarmCategory,
    HarmBlockThreshold) 
from google.cloud import storage

# For data handling.
import json
import jsonlines
import pandas as pd
from sklearn.model_selection import train_test_split



In [24]:
PROJECT_ID = "proj-sales-recommender-dev"
LOCATION = "us-central1" 
BUCKET = "sales_recommender_dev_bucket"
DATASET_NAME = "labeled_df_relevance.csv"

# Initialize Vertex AI and GenAI clients
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client(PROJECT_ID)

In [25]:
prompt = '''

    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.

        **Instructions:**

        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect project is provided in **Project Data**.

        2. **Utilize Search Terms:** Identify relevant products, materials, and phrases. The search terms are provided in the **Search Terms** section.

        3. **Consider Project Types:** Identify project type. Determine if the project is specialized and has high opportunity for work and visibility. 
        Examples of specialized projects are: 
            * Hospital and health services
            * Churches
            * Commercial real estate
            * Large residential apartments/dormitories
            * University/College buildings
            * Auditoriums
            * Senior living homes
        Examples of non-specialized projects with very low priority are:
            * One-time projects
            * Small residential projects
            * Golf courses

        4. **Identify Building Type**: Identify building type, including interior complexity and specialized work. 

        5. **Identify Locations and Distance:** Identify the project location and distance from nearest branch. Consider if a branch is too far away from a location. 
        Urban areas should have closer branches, while rural areas can have branches further away.

        6. **Identify Associated Brands:** Identify associated brands to the product. Associated brands include: 
            * Armstrong Ceilings 
            * Sto 
            * Dryvit

        7. **Identify Available Plans:** Identify if the project has detailed and available plans and specs.

        8. **Classify Projects:**
            a. Prioritize projects based on how relevant the inputs are to the search terms.
            b. Next, prioritize projects based on the project type, as specified in the previous steps. Deprioritize non-specialized projects. 
            c. Next, prioritize building types based on how complex the interior work is, as specified in the previous steps. Deprioritize projects with little interior work.
            d. Next, prioritize projects that have reasonable distance to the nearest branch, as specified in the previous steps. Deprioritize projects that are too far away from a branch.
            e. Next, prioritize projects that have associated brands, as specified in the previous steps. Lack of associated brands will not lower the priority.
            f. Next, increase priority if the project has detailed plans and specs. Lack of plans and specs will not lower the priority. 
            g. When other factors are equal, prioritize higher-value projects (e.g., higher total dollar amount).

        8. **Estimate Relevancy:** Estimate the relevancy of each project based on the above factors and total dollar amount.

        9. Respond in valid JSON: Return only in valid JSON format. The JSON should contain the classification of the project as very high, high, moderate, low, or not relevant. 
        ''' 

In [29]:
# Read df 
df = pd.read_csv(DATASET_NAME)

# Preprocess 
relevance_col_name = "Relevance"
df.dropna(subset=[relevance_col_name], inplace=True)
df.drop(columns = ['Query'], inplace = True)

# Split data
X = df.drop(columns=[relevance_col_name])
y = df[[relevance_col_name]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

# Combine ProjectID with y_train and y_test
y_train = pd.concat([pd.DataFrame(X_train['ProjectID']), y_train], axis = 1)
y_test = pd.concat([pd.DataFrame(X_test['ProjectID']), y_test], axis = 1)

In [41]:
def row_to_embedded_prompt(input_row, prompt, boolean_filters, examples: bool = True): 
    
    product = input_row['Search']
    input_row = input_row.drop('Search')
    search_terms = [filter['Query'] for filter in boolean_filters if filter['Filter'] == product][0]

    row_json_str = input_row.to_json()
    cc_project_json_record = json.loads(row_json_str)

    prompt += f'''

    **Input Data:**

    **Search:**
    {product}

    **Project Data:**
    {cc_project_json_record}

    **Search Terms:**
    {search_terms} 

    \n 
    '''
    
    if examples: 
        prompt += f'''**Example Output:** \n
      [
        {{"ProjectID": 1837563703, "Relevance": "Not Relevant"}}
        {{"ProjectID": 1837563704, "Relevance": "Low"}}
        {{"ProjectID": 1006193701, "Relevance": "Moderate"}}
        {{"ProjectID": 1837563705, "Relevance": "High"}}
        {{"ProjectID": 1006193702, "Relevance": "Very High"}}
      ]
        '''

    return prompt

In [47]:

boolean_filters = pd.read_csv(f"gs://{BUCKET}/data/boolean_filters_latest.csv")
boolean_filters = json.loads(boolean_filters.to_json(orient='records'))

# Prompt embedding 
lines = []
for index, row in X_train.iterrows():

    # Convert the row to JSON format
    input_json = row_to_embedded_prompt(row, prompt, boolean_filters)

    # Get the corresponding output from y_train
    output_json = y_train.loc[index].to_json()

    # Create the content row
    content_row = {}
    content_row["contents"] = []
    content_row["contents"].append({"role": "user", "parts": [{"text": input_json}]})
    content_row["contents"].append({"role": "model", "parts": [{"text": output_json}]})

    # Append the content row to the lines list
    lines.append(content_row)
lines[0]

{'contents': [{'role': 'user',
   'parts': [{'text': '\n\n    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.\n\n        **Instructions:**\n\n        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect project is provided in **Project Data**.\n\n        2. **Utilize Search Terms:** Identify relevant products, materials, and phrases. The search terms are provided in the **Search Terms** section.\n\n        3. **Consider Project Types:** Identify project type. Determine if the project is specialized and has high opportunity for work and visibility. \n        Examples of specialized projects are: \n            * Hospital and health services\n            * Churches\n            * Commercial real estate\n            * Large residential apartments/dormitories\n            * University/College buildings\n            * Auditor

In [48]:
from IPython.display import Markdown, display
display(Markdown(lines[0]["contents"][0]["parts"][0]["text"]))



    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.

        **Instructions:**

        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect project is provided in **Project Data**.

        2. **Utilize Search Terms:** Identify relevant products, materials, and phrases. The search terms are provided in the **Search Terms** section.

        3. **Consider Project Types:** Identify project type. Determine if the project is specialized and has high opportunity for work and visibility. 
        Examples of specialized projects are: 
            * Hospital and health services
            * Churches
            * Commercial real estate
            * Large residential apartments/dormitories
            * University/College buildings
            * Auditoriums
            * Senior living homes
        Examples of non-specialized projects with very low priority are:
            * One-time projects
            * Small residential projects
            * Golf courses

        4. **Identify Building Type**: Identify building type, including interior complexity and specialized work. 

        5. **Identify Locations and Distance:** Identify the project location and distance from nearest branch. Consider if a branch is too far away from a location. 
        Urban areas should have closer branches, while rural areas can have branches further away.

        6. **Identify Associated Brands:** Identify associated brands to the product. Associated brands include: 
            * Armstrong Ceilings 
            * Sto 
            * Dryvit

        7. **Identify Available Plans:** Identify if the project has detailed and available plans and specs.

        8. **Classify Projects:**
            a. Prioritize projects based on how relevant the inputs are to the search terms.
            b. Next, prioritize projects based on the project type, as specified in the previous steps. Deprioritize non-specialized projects. 
            c. Next, prioritize building types based on how complex the interior work is, as specified in the previous steps. Deprioritize projects with little interior work.
            d. Next, prioritize projects that have reasonable distance to the nearest branch, as specified in the previous steps. Deprioritize projects that are too far away from a branch.
            e. Next, prioritize projects that have associated brands, as specified in the previous steps. Lack of associated brands will not lower the priority.
            f. Next, increase priority if the project has detailed plans and specs. Lack of plans and specs will not lower the priority. 
            g. When other factors are equal, prioritize higher-value projects (e.g., higher total dollar amount).

        8. **Estimate Relevancy:** Estimate the relevancy of each project based on the above factors and total dollar amount.

        9. Respond in valid JSON: Return only in valid JSON format. The JSON should contain the classification of the project as very high, high, moderate, low, or not relevant. 
        

    **Input Data:**

    **Search:**
    EIFS & Stucco Sales

    **Project Data:**
    {'ProjectID': 1004451106, 'Title': 'Broadway and Bayaud Mixed-Use Development', 'Stage': 'General Contractor Award', 'Valuation_Value': 50000000, 'Valuation_Currency': 'USD', 'Parameters_Parameter_Ownership': 'Private', 'Parameters_Parameter_WorkType': 'New', 'Parameters_Parameter_Structures': 1.0, 'DocumentAvailability_Plans': False, 'DocumentAvailability_Specs': False, 'DocumentAvailability_Addenda': False, 'ParentCategories_PrimaryCategoryName': 'Apartments', 'ParentCategories_ParentCategory': 'Category: RESIDENTIAL, Subcategories: Apartments', 'Addresses_Address': "[{'_ProjectAddressType': 'Project', 'ns0:AddressLine1': '99 S Broadway', 'ns0:AddressLine2': None, 'ns0:City': 'Denver', 'ns0:CountryRegion': 'UNITED STATES', 'ns0:County': 'Denver', 'ns0:Latitude': Decimal('39.704419000'), 'ns0:Longitude': Decimal('-104.959341000'), 'ns0:StateProvince': 'CO', 'ns0:ZipPostalCode': '80209'}]", 'Details_Detail_Scope': "['Site work, paving and new construction of a multi-residential development in Denver, Colorado. Completed plans call for the construction of a three-story above grade, 232-unit multi-residential development.\\nThe scope of work includes the construction of a mixed-use development to include ground floor retail with a mix of studios, one and two bedroom units, along with townhomes and parking. As of August 1, 2018, the project is under construction. A general contractor has been selected. Construction began in July 2018.* *Project information, including timeline and contacts, has been obtained through public sources. The content management team continues to pursue additional details; however, the contact(s) listed have yet to disclose or confirm any information. Inquiries should be directed to the contact(s) listed.']", 'Details_Detail_Notes': "['Development include(s):  Site Work, Paving, New Construction']", 'Details_Detail': "[{'_': 'Site work, paving and new construction of a multi-residential development in Denver, Colorado. Completed plans call for the construction of a three-story above grade, 232-unit multi-residential development.\\nThe scope of work includes the construction of a mixed-use development to include ground floor retail with a mix of studios, one and two bedroom units, along with townhomes and parking. As of August 1, 2018, the project is under construction. A general contractor has been selected. Construction began in July 2018.* *Project information, including timeline and contacts, has been obtained through public sources. The content management team continues to pursue additional details; however, the contact(s) listed have yet to disclose or confirm any information. Inquiries should be directed to the contact(s) listed.', '_DetailType': 'Scope'}\n {'_': 'Development include(s):  Site Work, Paving, New Construction', '_DetailType': 'Notes'}]", 'RSMeansMaterialDivisions_Division_Metals': None, 'RSMeansMaterialDivisions_Division_ThermalandMoistureProtection': 'Name: Thermal Insulation, Code: 721, Installation Cost: 166596.28, Material Cost: 274645.0, Total Cost: 441241.28\nName: Weather Barriers, Code: 725, Installation Cost: 31865.73, Material Cost: 14484.42, Total Cost: 46350.15\nName: Vapor Retarders, Code: 726, Installation Cost: 13976.2, Material Cost: 5082.25, Total Cost: 19058.45\nName: Shingles And Shakes, Code: 731, Installation Cost: 77504.37, Material Cost: 102915.64, Total Cost: 180420.01\nName: Flexible Flashing, Code: 765, Installation Cost: 313371.78, Material Cost: 204357.43, Total Cost: 517729.21', 'RSMeansMaterialDivisions_Division_Openings': 'Name: Metal Frames, Code: 812, Installation Cost: 19645.03, Material Cost: 75729.82, Total Cost: 95374.85\nName: Metal Doors, Code: 813, Installation Cost: 21389.94, Material Cost: 245981.09, Total Cost: 267371.03\nName: Wood Doors, Code: 814, Installation Cost: 118841.0, Material Cost: 154922.12, Total Cost: 273763.12\nName: Metal Windows, Code: 851, Installation Cost: 256145.6, Material Cost: 894476.69, Total Cost: 1150622.29\nName: Door Hardware, Code: 871, Installation Cost: 115085.23, Material Cost: 506162.91, Total Cost: 621248.14', 'RSMeansMaterialDivisions_Division_Finishes': 'Name: Gypsum Board, Code: 929, Installation Cost: 1870171.19, Material Cost: 767847.25, Total Cost: 2638018.44\nName: Tiling, Code: 930, Installation Cost: 245430.51, Material Cost: 303537.62, Total Cost: 548968.13\nName: Resilient Flooring, Code: 965, Installation Cost: 32475.6, Material Cost: 70897.44, Total Cost: 103373.04\nName: Carpeting, Code: 968, Installation Cost: 145682.81, Material Cost: 1060297.93, Total Cost: 1205980.74\nName: Acoustic Insulation, Code: 981, Installation Cost: 72845.64, Material Cost: 88092.4, Total Cost: 160938.04\nName: Painting, Code: 991, Installation Cost: 520619.94, Material Cost: 205573.41, Total Cost: 726193.35', 'RSMeansMaterialDivisions_Division_Masonry': 'Name: Common Work Results For Masonry, Code: 405, Installation Cost: 40658.03, Material Cost: 17889.53, Total Cost: 58547.56\nName: Stone Masonry, Code: 443, Installation Cost: 1992243.55, Material Cost: 1943453.91, Total Cost: 3935697.46', 'Details_Detail_Details': '[]', 'Materials_Material': '[]', 'Notes_Note': '[]', 'Latitude': 39.704419, 'Longitude': -104.959341, 'closest_branch': 4.4}

    **Search Terms:**
    (sto NEAR eifs) OR (sto NEAR stucco) OR (stotique NEAR eifs) OR (stotique NEAR stucco) OR (essence NEAR eifs) OR (essence NEAR stucco) OR ('rapid guard' NEAR eifs) OR ('rapid guard' NEAR stucco) OR (stoguard NEAR eifs) OR (stoguard NEAR stucco) OR (STOLIT) OR (LOTUSAN) OR (StoCoat) OR ('STO BTS-PLUS') OR ('PRIMER / ADHESIVE-B') OR ('NEW GOLD COAT') OR ('FINE SAND FINISH') OR ('WATERTIGHT COAT-GREY') OR ('STO MESH') OR ('DETAIL MESH') OR ('STOGUARD MESH') OR ('STOGUARD FABRIC') OR ('ARMOR MAT') OR ('STO BTS-FASTSET DRY') OR ('FREEFORM TSW') OR ('FINE SAND') OR ('RAPID GUARD') OR ('BTS-PLUS') OR ('MEDIUM SAND') OR ('FREEFORM') OR ('LOTUSAN FREEFORM') OR ('SWIRL FINISH') OR (DRAINSCREEN) OR (STOSEAL) OR ('THERMO DOWEL') OR ('THERMO CAP') OR ('THERMO COUTERSUNK') OR ('DRAINSCREEN 6mm MAT') OR (adex NEAR eifs) OR (adex NEAR stucco) OR (dryvit NEAR eifs) OR (dryvit NEAR stucco) OR (parex NEAR eifs) OR (parex NEAR stucco) OR ('SPEC MIX SCRATCH & BROWN') OR (MTI NEAR stucco) OR ('WEEP SCREED') OR ('LATH SELF FURRING') OR (GRAVITY CAVITY) OR ('EZ CASING BD') OR (LATH) OR (AMICO) OR ('VAPORSEAL-R') OR ('CONFORMABLE MESH') OR ('EXP FLG CASING BEAD') OR (EPS FOAM) OR ('STO PRIMER') OR (TYVEK) OR (NICHIHA) OR (JHT) OR ('STO PRIMER / ADHESIVE-B') OR ('ESSENCE FINE SAND') OR ('ESSENCE SWIRL FINISH') OR ('STOCOAT NEW GOLD COAT') OR ('TRIMTEX DUAL ANGLE SAND') OR ('IPG 48MMx55M RED STUCCO') OR ('VINTAGE WOOD CEDAR') OR ('ULTIMATE CLIP II') OR (TYVEKFT475) OR (TYVEKTP3) OR (TYVEKFLEX975) OR (GCPAT) OR ('ICE & WATER') OR ('TAMLYN XTREME') 

    
 
    **Example Output:** 

      [
        {"ProjectID": 1837563703, "Relevance": "Not Relevant"}
        {"ProjectID": 1837563704, "Relevance": "Low"}
        {"ProjectID": 1006193701, "Relevance": "Moderate"}
        {"ProjectID": 1837563705, "Relevance": "High"}
        {"ProjectID": 1006193702, "Relevance": "Very High"}
      ]
        

In [51]:
# Write the lines to a JSONL file
output_file_embedded = "training_data_for_tuning_job_embedded.jsonl"
with open(output_file_embedded, 'w') as f:
    for line in lines:
        f.write(json.dumps(line) + '\n')

# Upload the file to GCS
bucket = storage_client.bucket(BUCKET)
blob = bucket.blob(f'data/tuning/{output_file_embedded}')
blob.upload_from_filename(output_file_embedded)

In [52]:
# Tune the model 
base_model = "gemini-2.0-flash-001"
tuned_model_name = "tuned-model-embedded"

from vertexai.tuning import sft 

sft_tuning_job_embedded = sft.train(
    source_model = base_model,
    train_dataset = f"gs://{BUCKET}/data/tuning/{output_file_embedded}",
    tuned_model_display_name = tuned_model_name, 
    epochs = 50
)

Creating SupervisedTuningJob
SupervisedTuningJob created. Resource name: projects/195063057478/locations/us-central1/tuningJobs/4303370055917240320
To use this SupervisedTuningJob in another session:
tuning_job = sft.SupervisedTuningJob('projects/195063057478/locations/us-central1/tuningJobs/4303370055917240320')
View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/us-central1/tuning/tuningJob/4303370055917240320?project=195063057478


In [ ]:
sft_tuning_job_embedded.refresh()
sft_tuning_job_embedded.tuned_model_name
tuned_model_embedded_endpoint = sft_tuning_job_embedded.tuned_model_endpoint_name

In [45]:
# Format testing data for prediction
X_test_prompts = []

for _, row in X_test.iterrows(): 
    input_json = row_to_embedded_prompt(row, prompt, boolean_filters, examples = True)
    X_test_prompts.append(input_json) 

display(Markdown(X_test_prompts[0]))



    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.

        **Instructions:**

        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect project is provided in **Project Data**.

        2. **Utilize Search Terms:** Identify relevant products, materials, and phrases. The search terms are provided in the **Search Terms** section.

        3. **Consider Project Types:** Identify project type. Determine if the project is specialized and has high opportunity for work and visibility. 
        Examples of specialized projects are: 
            * Hospital and health services
            * Churches
            * Commercial real estate
            * Large residential apartments/dormitories
            * University/College buildings
            * Auditoriums
            * Senior living homes
        Examples of non-specialized projects with very low priority are:
            * One-time projects
            * Small residential projects
            * Golf courses

        4. **Identify Building Type**: Identify building type, including interior complexity and specialized work. 

        5. **Identify Locations and Distance:** Identify the project location and distance from nearest branch. Consider if a branch is too far away from a location. 
        Urban areas should have closer branches, while rural areas can have branches further away.

        6. **Identify Associated Brands:** Identify associated brands to the product. Associated brands include: 
            * Armstrong Ceilings 
            * Sto 
            * Dryvit

        7. **Identify Available Plans:** Identify if the project has detailed and available plans and specs.

        8. **Classify Projects:**
            a. Prioritize projects based on how relevant the inputs are to the search terms.
            b. Next, prioritize projects based on the project type, as specified in the previous steps. Deprioritize non-specialized projects. 
            c. Next, prioritize building types based on how complex the interior work is, as specified in the previous steps. Deprioritize projects with little interior work.
            d. Next, prioritize projects that have reasonable distance to the nearest branch, as specified in the previous steps. Deprioritize projects that are too far away from a branch.
            e. Next, prioritize projects that have associated brands, as specified in the previous steps. Lack of associated brands will not lower the priority.
            f. Next, increase priority if the project has detailed plans and specs. Lack of plans and specs will not lower the priority. 
            g. When other factors are equal, prioritize higher-value projects (e.g., higher total dollar amount).

        8. **Estimate Relevancy:** Estimate the relevancy of each project based on the above factors and total dollar amount.

        9. Respond in valid JSON: Return only in valid JSON format. The JSON should contain the classification of the project as very high, high, moderate, low, or not relevant. 
        

    **Input Data:**

    **Search:**
    Steel Sales

    **Project Data:**
    {'ProjectID': 1004343343, 'Title': 'Park View at Palmetto Bay', 'Stage': 'Construction Underway', 'Valuation_Value': 43000000, 'Valuation_Currency': 'USD', 'Parameters_Parameter_Ownership': 'Private', 'Parameters_Parameter_WorkType': 'New', 'Parameters_Parameter_Structures': 2.0, 'DocumentAvailability_Plans': False, 'DocumentAvailability_Specs': False, 'DocumentAvailability_Addenda': False, 'ParentCategories_PrimaryCategoryName': 'Apartments', 'ParentCategories_ParentCategory': 'Category: CIVIL, Subcategories: Swimming Pools\nCategory: COMMUNITY, Subcategories: Clubs, Community Centers\nCategory: RESIDENTIAL, Subcategories: Apartments\nCategory: RETAIL, Subcategories: Retail Stores', 'Addresses_Address': "[{'_ProjectAddressType': 'Project', 'ns0:AddressLine1': '9500 SW 174th St', 'ns0:AddressLine2': None, 'ns0:City': 'Palmetto Bay', 'ns0:CountryRegion': 'UNITED STATES', 'ns0:County': 'Miami-Dade', 'ns0:Latitude': Decimal('25.606686000'), 'ns0:Longitude': Decimal('-80.344992000'), 'ns0:StateProvince': 'FL', 'ns0:ZipPostalCode': '33157'}]", 'Details_Detail_Scope': "['Site work and new construction of a mixed-use development in Palmetto Bay, Florida. Completed plans call for the construction of a 235-unit multi-residential development; and retail development.\\nAll trades have been secured.']", 'Details_Detail_Notes': "['Municipal Meeting: 04/17/2017 07:00PM Village of Palmetto Bay Zoning Hearing\\nDevelopment include(s):  New Construction, Site Work']", 'Details_Detail': "[{'_': 'Site work and new construction of a mixed-use development in Palmetto Bay, Florida. Completed plans call for the construction of a 235-unit multi-residential development; and retail development.\\nAll trades have been secured.', '_DetailType': 'Scope'}\n {'_': 'Municipal Meeting: 04/17/2017 07:00PM Village of Palmetto Bay Zoning Hearing\\nDevelopment include(s):  New Construction, Site Work', '_DetailType': 'Notes'}]", 'RSMeansMaterialDivisions_Division_Metals': 'Name: Structural Steel Framing, Code: 512, Installation Cost: 153267.12, Material Cost: 746666.93, Total Cost: 899934.05\nName: Structural Aluminum Framing, Code: 514, Installation Cost: 7559.48, Material Cost: 25198.28, Total Cost: 32757.76\nName: Steel Joist Framing, Code: 521, Installation Cost: 229344.02, Material Cost: 739244.79, Total Cost: 968588.81\nName: Steel Decking, Code: 531, Installation Cost: 131928.48, Material Cost: 395228.77, Total Cost: 527157.25\nName: Metal Stairs, Code: 551, Installation Cost: 92638.78, Material Cost: 728986.89, Total Cost: 821625.67', 'RSMeansMaterialDivisions_Division_ThermalandMoistureProtection': 'Name: Roof And Deck Insulation, Code: 722, Installation Cost: 6865.48, Material Cost: 37481.79, Total Cost: 44347.27\nName: Vapor Retarders, Code: 726, Installation Cost: 2041.09, Material Cost: 742.21, Total Cost: 2783.3\nName: Built-Up Bituminous Roofing, Code: 751, Installation Cost: 873.88, Material Cost: 1527.17, Total Cost: 2401.05\nName: Elastomeric Membrane Roofing, Code: 753, Installation Cost: 6733.03, Material Cost: 19757.13, Total Cost: 26490.16\nName: Flexible Flashing, Code: 765, Installation Cost: 1951.38, Material Cost: 1060.53, Total Cost: 3011.91\nName: Roof Specialties, Code: 771, Installation Cost: 5557.2, Material Cost: 12005.24, Total Cost: 17562.44\nName: Joint Sealants, Code: 792, Installation Cost: 118431.92, Material Cost: 49136.65, Total Cost: 167568.57', 'RSMeansMaterialDivisions_Division_Openings': 'Name: Metal Doors And Frames, Code: 811, Installation Cost: 6526.36, Material Cost: 18811.28, Total Cost: 25337.64\nName: Wood Doors, Code: 814, Installation Cost: 90311.31, Material Cost: 120771.91, Total Cost: 211083.22\nName: Sliding Glass Doors, Code: 832, Installation Cost: 68705.31, Material Cost: 584349.28, Total Cost: 653054.59\nName: Entrances And Storefronts, Code: 841, Installation Cost: 928556.66, Material Cost: 2017122.41, Total Cost: 2945679.07\nName: Door Hardware, Code: 871, Installation Cost: 77222.7, Material Cost: 327132.54, Total Cost: 404355.24\nName: Glass Glazing, Code: 881, Installation Cost: 789461.78, Material Cost: 809247.31, Total Cost: 1598709.09', 'RSMeansMaterialDivisions_Division_Finishes': 'Name: Supports For Plaster And Gypsum Board, Code: 922, Installation Cost: 102920.29, Material Cost: 45522.44, Total Cost: 148442.73\nName: Gypsum Board, Code: 929, Installation Cost: 1263852.56, Material Cost: 521917.74, Total Cost: 1785770.3\nName: Tiling, Code: 930, Installation Cost: 179213.67, Material Cost: 221643.55, Total Cost: 400857.22\nName: Resilient Flooring, Code: 965, Installation Cost: 23713.73, Material Cost: 51769.4, Total Cost: 75483.13\nName: Carpeting, Code: 968, Installation Cost: 106377.77, Material Cost: 774230.89, Total Cost: 880608.66\nName: Painting, Code: 991, Installation Cost: 274303.74, Material Cost: 103569.08, Total Cost: 377872.82', 'RSMeansMaterialDivisions_Division_Masonry': 'Name: Common Work Results For Masonry, Code: 405, Installation Cost: 16823.51, Material Cost: 22761.22, Total Cost: 39584.73\nName: Concrete Unit Masonry, Code: 422, Installation Cost: 441369.71, Material Cost: 316677.82, Total Cost: 758047.53', 'Details_Detail_Details': '[]', 'Materials_Material': '[]', 'Notes_Note': '[]', 'Latitude': 25.606686, 'Longitude': -80.344992, 'closest_branch': 15.72}

    **Search Terms:**
    (steel) OR (stud) OR (track) OR (angle) OR ('flat stock') OR (channel) OR (runner) OR (furring) OR (spazzer) OR (resilient) OR ('COLD ROLLED') OR ('Z FURRING') OR ('CT STUD') OR ('J TABBED TRACK') OR (SLOTTED) OR (FASTCLIP) OR (SUBH) OR (SIMPSON) OR ('Pony Wall') OR (ShortSpan) OR ('Drywall Furring Channel') OR ('ROCKSTEADY') OR ('SCORPION HT') OR ('GEMCO INSULATION') OR ('EQ 18M') OR ('SECURITY MESH') OR ('BARRIER MESH CLIP') OR ('JOISTRITE') OR ('SECURA CLIP') OR ('DW STD') OR ('DW TRK') OR ('FLEX C') OR (UNPUNCHED) OR ('UTILITY CLIP') OR ('PONY WALL') OR ('SPAZZER BAR') OR ('STEEL CLIPS') OR ('DW FURRING') OR ('FLAT STOCK') OR ('C RUNNER') OR ('TIE WIRE') OR ('STR STUD') OR ('RADIUS TRACK') OR ('COLD ROLLED CHANNEL') OR (REDHDR) OR ('ANCHOR CLIP') OR ('LATH SELF FURRING') OR ('WEEP SCREED') OR ('FIRE TRAK') OR ('FIRE TRAK') OR ('STUDRITE') OR ('WIRE MESH') OR ('Z-GIRT') OR (TRImaco) OR ('T/R JOIST') OR ('FLAT BUGLE WASHER') OR ('GRIP-DECK CI CERAMIC') OR ('KENBECK') OR ('GORDON 901') OR ('ANGLE CLIP') OR ('COLD ROLLED CHANNEL') OR (Metaltech) OR ('PONY WALL') OR ('PONY WALL LITE') OR ('TRA LOC') 

    
 
    **Example Output:** 

      [
        {"ProjectID": 1837563703, "Relevance": "Not Relevant"}
        {"ProjectID": 1837563704, "Relevance": "Low"}
        {"ProjectID": 1006193701, "Relevance": "Moderate"}
        {"ProjectID": 1837563705, "Relevance": "High"}
        {"ProjectID": 1006193702, "Relevance": "Very High"}
      ]
        

In [ ]:
# Get tuned and based
from vertexai.generative_models import GenerativeModel, GenerationConfig 

response_schema = {
        "type": "OBJECT",
        "properties": {
            "ProjectID": {"type": "INTEGER"},
            "Relevance": {"type": "STRING"}
        }
}

config = GenerationConfig(response_schema=response_schema, response_mime_type="application/json")

MODEL_ID = "gemini-2.0-flash-001"

tuned_model = GenerativeModel(tuned_model_embedded_endpoint)
base_model = GenerativeModel(MODEL_ID)

tuned_predictions = [] 
base_predictions = []
for prompt in tqdm(X_test_prompts): 
    tuned_response = tuned_model.generate_content(contents = [prompt], generation_config = config)
    tuned_predictions.append(tuned_response.text)

    base_response = base_model.generate_content(contents = [prompt], generation_config = config)
    base_predictions.append(base_response.text)

100%|██████████| 30/30 [00:50<00:00,  1.69s/it]


In [19]:
def strip_json_response(response):
    output = response.replace("```json", "").replace("```", "").replace("[", "").replace("]", "").replace("\n","").strip() 
    output = json.loads(output)
    return output['Relevance']

cleaned_tuned_predictions = [strip_json_response(pred) for pred in tuned_predictions]
cleaned_based_predictions = [strip_json_response(pred) for pred in base_predictions]

predictions_df = y_test.copy() 
predictions_df['Tuned Relevance'] = cleaned_tuned_predictions
predictions_df['Base Relevance'] = cleaned_based_predictions
predictions_df.head()

,ProjectID,Relevance,Tuned Relevance,Base Relevance
109,1004206072,High,Very High,Moderate
14,1002057138,Very High,Very High,High
107,1004182711,Very High,Very High,Not Relevant
122,1004311228,Very High,High,Moderate
62,1002464636,High,Very High,Moderate


In [20]:

from sklearn.preprocessing import OrdinalEncoder 
relevance_columns = predictions_df.filter(regex='Relevance').columns

# Encode the relevance columns
labels = ['not relevant', 'low', 'moderate', 'high', 'very high']
encoder = OrdinalEncoder(categories=[labels])

for col in relevance_columns: 
    predictions_df[col] = predictions_df[col].astype(str).str.lower()
    predictions_df[col] = encoder.fit_transform(predictions_df[[col]])



In [21]:
from sklearn.metrics import f1_score

print(f"F1 Score for {col}: ", f1_score(predictions_df['Relevance'], predictions_df["Base Relevance"], average='weighted'))
print(f"F1 Score for Tuned Predictions: ", f1_score(predictions_df['Relevance'], predictions_df["Tuned Relevance"], average='weighted'))

F1 Score for Base Relevance:  0.3490374331550802
F1 Score for Tuned Predictions:  0.3894685990338164
